# Chapter 46: Vector Retrieval — Retrieving Nearest Documents

In Chapter 45, you stored documents, IDs, metadata, and embeddings in ChromaDB. Now we'll focus on retrieving the most relevant documents for a query.
The goal is to understand how query embeddings are compared with stored vectors, how ChromaDB ranks results, and how to interpret the distances returned.
This notebook is fully independent: it creates its own dataset, embedding model, and in-memory ChromaDB collection. It does not rely on variables or files from Chapter 45.

# Step 1 — Install the dependencies

In [ ]:
%pip install chromadb sentence-transformers numpy

# Step 2 — Import the libraries

In [2]:
import chromadb
import numpy as np
from sentence_transformers import SentenceTransformer

/Users/vimal/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


# Step 3 — Create a standalone dataset

In [3]:
documents = [
    "Banks monitor cross-border payments for suspicious activity.",
    "Money laundering involves hiding the illegal origin of funds.",
    "Customer due diligence verifies customer identities.",
    "Credit risk models estimate the probability of borrower default.",
    "Machine learning identifies unusual financial transaction patterns.",
    "Fraud detection systems identify potentially fraudulent payments.",
    "Neural networks are used for image classification.",
    "Payment processing systems transfer money between accounts."
]

ids = [f"doc_{i}" for i in range(1, len(documents) + 1)]

metadatas = [
    {"category": "AML"},
    {"category": "AML"},
    {"category": "KYC"},
    {"category": "Risk"},
    {"category": "AML"},
    {"category": "Fraud"},
    {"category": "AI"},
    {"category": "Payments"}
]

print("Documents:", len(documents))

Documents: 8


# Step 4 — Generate document embeddings

In [4]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

document_embeddings = model.encode(documents)


print("Embedding shape:", document_embeddings.shape)

Embedding shape: (8, 384)


# Step 5 — Create ChromaDB and store the vectors

In [5]:
client = chromadb.EphemeralClient()

collection = client.create_collection(
    name="chapter_46_retrieval",
    metadata={"hnsw:space": "cosine"}
)

In [7]:
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=document_embeddings)



In [8]:
collection.count()

8

# Step 6 — Convert a query into an embedding

In [9]:
query = "How can banks identify suspicious international transfers?"

In [10]:
query_embedding = model.encode(query)

In [11]:
query_embedding

array([ 3.33284587e-02, -4.17652130e-02, -4.94390316e-02, -6.33582249e-02,
        9.37114730e-02, -7.41584972e-02,  1.51743010e-01, -6.84304386e-02,
        7.12414458e-02, -3.55836339e-02, -7.00040767e-03, -3.20354290e-02,
        7.27062533e-03,  4.89794686e-02, -1.04044311e-01, -6.86592683e-02,
       -4.98563461e-02, -1.95983555e-02, -2.22064997e-03, -2.25586705e-02,
       -1.47693232e-02, -6.89662695e-02, -2.52827946e-02, -1.14143360e-02,
        4.38107736e-02,  3.37828696e-03,  1.76022165e-02, -1.83926895e-02,
       -7.45847747e-02, -3.74853127e-02, -2.02308744e-02,  3.99743132e-02,
       -1.04203507e-01, -1.82670187e-02,  4.39735949e-02, -5.90110235e-02,
        5.59986234e-02,  8.04737769e-03,  8.89882818e-02, -5.51890396e-02,
        4.53554429e-02, -1.08348489e-01,  1.32558137e-01,  2.01887693e-02,
       -3.24667394e-02,  6.86049163e-02,  3.67171317e-03,  7.33310878e-02,
       -3.28759365e-02,  2.51915324e-02,  2.43288726e-02, -2.05808077e-02,
       -1.48457810e-02,  

In [13]:
query_embedding.shape

(384,)

# Step 7 — Retrieve the nearest documents

In [17]:
results = collection.query(
    query_embeddings=query_embedding,
    n_results=3)

In [18]:
results

{'ids': [['doc_1', 'doc_6', 'doc_2']],
 'embeddings': None,
 'documents': [['Banks monitor cross-border payments for suspicious activity.',
   'Fraud detection systems identify potentially fraudulent payments.',
   'Money laundering involves hiding the illegal origin of funds.']],
 'uris': None,
 'included': ['metadatas', 'documents', 'distances'],
 'data': None,
 'metadatas': [[{'category': 'AML'},
   {'category': 'Fraud'},
   {'category': 'AML'}]],
 'distances': [[0.21935439109802246,
   0.45328229665756226,
   0.46831512451171875]]}

In [19]:
for i, document in enumerate(results["documents"][0]):
    print(f"\nRank: {i + 1}")
    print("Document:", document)
    print("Category:", results["metadatas"][0][i]["category"])
    print("Distance:", round(results["distances"][0][i], 4))


Rank: 1
Document: Banks monitor cross-border payments for suspicious activity.
Category: AML
Distance: 0.2194

Rank: 2
Document: Fraud detection systems identify potentially fraudulent payments.
Category: Fraud
Distance: 0.4533

Rank: 3
Document: Money laundering involves hiding the illegal origin of funds.
Category: AML
Distance: 0.4683


The query asks about suspicious international transfers, so documents about cross-border payments, transaction monitoring, and suspicious payment behavior may rank highly.
The exact ordering depends on the embedding model and the text. Similarity does not guarantee that a result is factually correct.

Because our collection uses cosine distance:
- A smaller distance means the vectors are closer in direction.
- A distance of 0 means the vectors have the same direction, assuming nonzero vectors.
- A distance score is not a percentage and does not indicate factual accuracy.